# Streaming (Table 5 + Table 6) — SplitFashionMNIST

Notebook riêng, tách ra để chỉ chạy phần **Streaming** của Mục 5.4 (paper Borsos et al.,
2024): so sánh Reservoir / CBRS / Coreset (BiCo) trên luồng dữ liệu cân bằng (**Table 5**),
và Reservoir / CBRS trên luồng mất cân bằng (**Table 6**), cả hai trên SplitFashionMNIST
(buffer=100, β=1.0).

**Dữ liệu FashionMNIST:** nếu bạn đã tải/chuẩn bị sẵn và upload thành một **Kaggle
Dataset** riêng, điền đường dẫn vào biến `FASHION_DATA_ROOT` ở Ô cấu hình bên dưới để
script dùng lại thay vì tải qua mạng mỗi phiên. Để trống (`None`) nếu muốn tải bình
thường qua `torchvision`.

**HƯỚNG DẪN:**
1. Bật GPU (T4 hoặc P100) trong Settings của Kaggle.
2. (Tuỳ chọn) Add Data → gắn Kaggle Dataset chứa FashionMNIST đã tải sẵn.
3. Chạy Ô Số 1 (clone) → **Restart Session** khi được yêu cầu.
4. Chạy tiếp các ô còn lại từ trên xuống.


### Ô Số 0: Kiểm tra GPU

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "KHONG CO GPU! Vao Settings > Accelerator, chon GPU (T4 x2 hoac P100), "
        "bam Save, roi chay lai notebook tu dau."
    )
print('GPU OK:', torch.cuda.get_device_name(0))


In [ ]:
# Ô SỐ 1: CÀI ĐẶT MÔI TRƯỜNG VÀ TẢI MÃ NGUỒN
!git clone https://github.com/quachthanhhmd/bilevel-coresets.git
%cd bilevel-coresets
!git checkout master


⚠️ **CẢNH BÁO QUAN TRỌNG:** Dừng lại tại đây! Bạn phải bấm `Restart Session` (hoặc `Restart Kernel`) trước khi chạy ô tiếp theo.

In [ ]:
# 1. Hạ cấp setuptools để vá lỗi môi trường build của Python 3.12
!pip install "setuptools<70.0.0"

# 2. Cài duy nhất thư viện mô phỏng mạng CNN còn thiếu
!pip install neural-tangents

!pip install --upgrade jax jaxlib==0.1.56+cuda101 -f https://storage.googleapis.com/jax-releases/jax_releases.html


In [ ]:
import jax
import jax.core
import jax._src.core
import jax.tree_util
import jax.util

# 1. Vá các class lõi bị giấu (Đã thêm Primitive)
missing_classes = [
    'Jaxpr', 'JaxprEqn', 'Literal', 'Var', 'DropVar',
    'ClosedJaxpr', 'ShapedArray', 'Value', 'MainTrace', 'Trace',
    'Primitive'  # <--- CHÍNH LÀ THỦ PHẠM MỚI NHẤT
]
for cls_name in missing_classes:
    if hasattr(jax._src.core, cls_name):
        setattr(jax.core, cls_name, getattr(jax._src.core, cls_name))

# 2. Vá hàm tree_multimap
if not hasattr(jax.tree_util, 'tree_multimap'):
    jax.tree_util.tree_multimap = jax.tree_util.tree_map

# 3. Vá hàm safe_map và safe_zip cho jax.util
def custom_safe_map(f, *args):
    return list(map(f, *args))

def custom_safe_zip(*args):
    return list(zip(*args))

jax.util.safe_map = custom_safe_map
jax.util.safe_zip = custom_safe_zip

print("Đã vá nóng JAX V4: Bổ sung Primitive! Sẵn sàng nạp Neural Tangents.")


### Ô Số 2: Cấu hình

**Nếu đã có Kaggle Dataset chứa FashionMNIST tải sẵn:** thêm dataset đó vào notebook
(nút *Add Input* / *Add Data* ở panel bên phải), rồi điền đúng đường dẫn vào
`FASHION_DATA_ROOT` bên dưới. Script tìm dữ liệu tại
`<FASHION_DATA_ROOT>/FashionMNIST/raw/...` — đây là quy ước gốc của `torchvision`, nên
dataset bạn upload cần giữ đúng cấu trúc thư mục này (thư mục con tên `FashionMNIST`,
bên trong có thư mục `raw` chứa 4 file `train-images-idx3-ubyte`,
`train-labels-idx1-ubyte`, `t10k-images-idx3-ubyte`, `t10k-labels-idx1-ubyte`, có hoặc
không nén `.gz`).

Để `FASHION_DATA_ROOT = None` nếu muốn để script tự tải qua mạng như bình thường.


In [ ]:
import os, subprocess, json

# Tự dò thư mục repo đã clone -- chạy được trên cả Kaggle lẫn Colab (và các nền tảng
# khác), không phụ thuộc đường dẫn /kaggle/working cố định.
_candidates = [
    '/kaggle/working/bilevel-coresets',
    '/content/bilevel-coresets',
    os.path.join(os.getcwd(), 'bilevel-coresets'),
]
REPO = next((c for c in _candidates if os.path.isdir(c)), None)
if REPO is None:
    raise RuntimeError(
        "Khong tim thay thu muc 'bilevel-coresets' o cac vi tri thong thuong.\n"
        "Kiem tra lai da chay xong O SO 1 (git clone) chua, hoac sua _candidates o tren "
        "cho dung duong dan thuc te tren nen tang ban dang dung."
    )
os.chdir(REPO)

CL_DIR = os.path.join(REPO, 'cl_streaming')
ENV = os.environ.copy()
ENV['PYTHONPATH'] = REPO + os.pathsep + ENV.get('PYTHONPATH', '')

# <-- ĐIỀN ĐƯỜNG DẪN DATASET (Kaggle Dataset / Colab Drive...) CỦA BẠN VÀO ĐÂY (hoặc để None) -->
FASHION_DATA_ROOT = None   # ví dụ Kaggle: '/kaggle/input/my-fashionmnist-cached'
                           # ví dụ Colab:  '/content/drive/MyDrive/fashionmnist-cached'

BUFFER_SIZE = 100
BETA = 1.0
SEEDS = [0, 1, 2]   # chạy nhiều seed lấy trung bình, giảm nhiễu (xem giải thích Ô Số 2)

print('REPO =', REPO)
print('CL_DIR =', CL_DIR)
print('FASHION_DATA_ROOT =', FASHION_DATA_ROOT)


### Ô Số 3: Kiểm tra đường dẫn dữ liệu (nếu có đặt FASHION_DATA_ROOT)

In [ ]:
if FASHION_DATA_ROOT:
    raw_dir = os.path.join(FASHION_DATA_ROOT, 'FashionMNIST', 'raw')
    if not os.path.isdir(raw_dir):
        raise RuntimeError(
            f"Khong tim thay thu muc {raw_dir}.\n"
            f"Kiem tra lai: (1) da 'Add Data' dataset chua trong notebook nay chua, "
            f"(2) FASHION_DATA_ROOT co dung ten slug dataset khong, "
            f"(3) dataset co giu dung cau truc <root>/FashionMNIST/raw/... khong."
        )
    found = sorted(os.listdir(raw_dir))
    print('Tìm thấy trong', raw_dir, ':', found)
    expected = ['train-images-idx3-ubyte', 'train-labels-idx1-ubyte',
                't10k-images-idx3-ubyte', 't10k-labels-idx1-ubyte']
    missing = [e for e in expected if not any(e in f for f in found)]
    if missing:
        print('CẢNH BÁO: có thể thiếu file (kể cả bản .gz):', missing,
              '-- torchvision sẽ thử tải qua mạng, có thể lỗi vì /kaggle/input chỉ đọc.')
    else:
        print('OK -- cấu trúc thư mục hợp lệ, sẽ dùng lại dữ liệu này, không tải qua mạng.')
else:
    print('FASHION_DATA_ROOT chưa được đặt -- sẽ tải FashionMNIST qua mạng như bình thường '
          '(lưu vào data/FashionMNIST trong phiên Kaggle hiện tại).')


## Table 5 — Streaming cân bằng (SplitFashionMNIST)

So sánh Reservoir / CBRS / Coreset (BiCo qua proxy Nyström-NTK) trên luồng dữ liệu cân
bằng. `coreset` cần jax/neural-tangents (đã cài ở Ô Số 1) để tính kernel proxy; `reservoir`
và `cbrs` không cần.

**Lưu ý:** chạy mỗi phương pháp trên {} seed khác nhau và lấy trung bình ± độ lệch chuẩn (giống giao thức 5 lần chạy của paper) -- kết quả 1 seed đơn lẻ có thể dao động vài điểm % do nhiều phép toán GPU không tất định, đặc biệt với `coreset` (chọn điểm dựa trên so sánh gradient rất nhạy với sai số dấu phẩy động).


In [ ]:
extra_args = ['--fashion_data_root', FASHION_DATA_ROOT] if FASHION_DATA_ROOT else []

for seed in SEEDS:
    for method in ['reservoir', 'cbrs', 'coreset']:
        print(f'--- streaming: splitfashionmnist / {method} / seed={seed} ---')
        subprocess.run(
            ['python', 'streaming.py',
             '--dataset', 'splitfashionmnist', '--method', method,
             '--seed', str(seed), '--buffer_size', str(BUFFER_SIZE), '--beta', str(BETA)]
            + extra_args,
            cwd=CL_DIR, env=ENV, check=True)
print('Table 5 (balanced) xong -- {} seed x 3 phương pháp.'.format(len(SEEDS)))


## Table 6 — Streaming mất cân bằng (SplitFashionMNIST)

4 task đầu chỉ giữ 200 điểm, task cuối giữ 2000 điểm. Paper chỉ so `reservoir` với `cbrs`
cho Table 6 (không có `coreset` ở bảng này), dùng `nr_slots=1`.


In [ ]:
for seed in SEEDS:
    for method in ['reservoir', 'cbrs', 'coreset']:
        print(f'--- streaming: splitfashionmnistimbalanced / {method} / seed={seed} ---')
        subprocess.run(
            ['python', 'streaming.py',
             '--dataset', 'splitfashionmnistimbalanced', '--method', method,
             '--seed', str(seed), '--buffer_size', str(BUFFER_SIZE), '--beta', str(BETA),
             '--nr_slots', '1']
            + extra_args,
            cwd=CL_DIR, env=ENV, check=True)
print('Table 6 (imbalanced) xong -- {} seed x 3 phương pháp.'.format(len(SEEDS)))


## Xem kết quả (Table 5 + Table 6)

In [ ]:
import numpy as np

def load_streaming_multiseed(dataset, methods, seeds, buffer_size=BUFFER_SIZE, beta=BETA):
    rows = []
    for method in methods:
        accs = []
        for seed in seeds:
            path = os.path.join(CL_DIR, 'streaming_results',
                                f'{dataset}_{method}_{buffer_size}_{beta}_{seed}.txt')
            if os.path.exists(path):
                with open(path) as f:
                    d = json.load(f)
                accs.append(d['test_acc'])
        rows.append((method, accs))
    return rows

print('=== Table 5: Streaming (balanced), trung bình {} seed ==='.format(len(SEEDS)))
for method, accs in load_streaming_multiseed('splitfashionmnist', ['reservoir', 'cbrs', 'coreset'], SEEDS):
    if accs:
        print(f'{method:12s} {np.mean(accs):.2f} ± {np.std(accs):.2f}   (n={len(accs)}, các lần: {["%.2f" % a for a in accs]})')
    else:
        print(f'{method:12s} chưa chạy')

print()
print('=== Table 6: Streaming (imbalanced), trung bình {} seed ==='.format(len(SEEDS)))
for method, accs in load_streaming_multiseed('splitfashionmnistimbalanced', ['reservoir', 'cbrs', 'coreset'], SEEDS):
    if accs:
        print(f'{method:12s} {np.mean(accs):.2f} ± {np.std(accs):.2f}   (n={len(accs)}, các lần: {["%.2f" % a for a in accs]})')
    else:
        print(f'{method:12s} chưa chạy')


## Đóng gói kết quả

In [ ]:
import shutil

_out_dir = os.path.dirname(REPO)  # thư mục cha của repo (=/kaggle/working hoặc /content...)
_zip_base = os.path.join(_out_dir, 'streaming_results_output')
zip_path = shutil.make_archive(_zip_base, 'zip', os.path.join(CL_DIR, 'streaming_results'))
print('Đã lưu', zip_path)
